In [1]:
import os
import scanpy as sc
import anndata as ad
import pandas as pd

In [2]:
import liana as li

In [6]:
import itertools

In [26]:
adata = sc.read_h5ad('/projects/bioinformatics/DB/scRNAseq_parkinson/dataset.h5ad')

In [4]:
lr_resource = pd.read_csv("/projects/bioinformatics/DB/CellCellCommunication/WithEnzymes/cpdb_cellchat_enz_liana_format.csv")

In [15]:
# we need to make sure that genes are not in the complex form but seperated G1 and G2
lr_resource[800:900]

,ligand,receptor
800,GAD1+SLC32A1,GABRA4_GABRB3_GABRD
801,GAD1+SLC32A1,GABRA4_GABRB3_GABRG2
802,GAD1+SLC32A1,GABRA5_GABRB3_GABRG2
803,GAD1+SLC32A1,GABRA6_GABRB3_GABRD
804,GAD1+SLC32A1,GABRA6_GABRB3_GABRG2
...,...,...
895,GAD1+SLC6A8,GABRA1_GABRB3
896,GAD1+SLC6A8,GABRA1_GABRB3_GABRG2
897,GAD1+SLC6A8,GABRA2_GABRB3_GABRG2
898,GAD1+SLC6A8,GABRA3_GABRB3_GABRE


In [16]:
import re

def split_genes(gene_entry):
    if pd.isna(gene_entry):
        return []
    
    # split on BOTH "_" and "+"
    genes = re.split(r"[+_]", str(gene_entry))
    
    # remove empty strings if any
    return [g for g in genes if g]

In [17]:
ligand_genes = lr_resource["ligand"].apply(split_genes)
receptor_genes = lr_resource["receptor"].apply(split_genes)

all_genes = set(itertools.chain.from_iterable(ligand_genes)) | \
            set(itertools.chain.from_iterable(receptor_genes))

print("Total unique LR genes:", len(all_genes))

Total unique LR genes: 1576


In [25]:
all_genes;

In [28]:
print(adata.var_names[:20])

Index(['ENSG00000186827', 'ENSG00000186891', 'ENSG00000160072',
       'ENSG00000041988', 'ENSG00000142611', 'ENSG00000067606',
       'ENSG00000131584', 'ENSG00000169972', 'ENSG00000157911',
       'ENSG00000224051', 'ENSG00000157933', 'ENSG00000162591',
       'ENSG00000169962', 'ENSG00000116285', 'ENSG00000142655',
       'ENSG00000149527', 'ENSG00000007923', 'ENSG00000171621',
       'ENSG00000162441', 'ENSG00000187017'],
      dtype='object', name='gene_id')


In [9]:
print(adata.var.head())

                gene_name  n_cells  feature_is_filtered feature_name  \
gene_id                                                                
ENSG00000186827   TNFRSF4     7846                False      TNFRSF4   
ENSG00000186891  TNFRSF18     9000                False     TNFRSF18   
ENSG00000160072    ATAD3B   340777                False       ATAD3B   
ENSG00000041988     THAP3   209923                False        THAP3   
ENSG00000142611    PRDM16   283396                False       PRDM16   

                feature_reference feature_biotype feature_length  \
gene_id                                                            
ENSG00000186827    NCBITaxon:9606            gene           1039   
ENSG00000186891    NCBITaxon:9606            gene            789   
ENSG00000160072    NCBITaxon:9606            gene           3300   
ENSG00000041988    NCBITaxon:9606            gene            931   
ENSG00000142611    NCBITaxon:9606            gene           3730   

                  

In [29]:
# Set gene symbols as var_names

adata.var_names = adata.var["gene_name"]

/home/znazari/.conda/envs/liana_scanpy/lib/python3.10/site-packages/anndata/_core/anndata.py:835: UserWarning: 
AnnData expects .var.index to contain strings, but got values like:
    ['TNFRSF4', 'TNFRSF18', 'ATAD3B', 'THAP3', 'PRDM16']

    Inferred to be: categorical



In [30]:
# to fix warning 
adata.var_names = adata.var["gene_name"].astype(str)

In [31]:
adata.var_names_make_unique()

In [32]:
print(adata.var_names[:10])
print(type(adata.var_names[0]))

Index(['TNFRSF4', 'TNFRSF18', 'ATAD3B', 'THAP3', 'PRDM16', 'PRKCZ', 'ACAP3',
       'PUSL1', 'PEX10', 'CPTP'],
      dtype='object', name='gene_name')
<class 'str'>


In [33]:
genes_in_data = list(set(adata.var_names) & all_genes)

print("Genes found in dataset:", len(genes_in_data))

Genes found in dataset: 1415


In [34]:
# subset 

adata_lr = adata[:, genes_in_data].copy()
print(adata_lr.shape)

(2096155, 1415)


In [ ]:
li.method.cellphonedb(
    adata_lr,
    groupby="cell_type",
    resource=lr_resource,
    use_raw=False
)

/home/znazari/.conda/envs/liana_scanpy/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/home/znazari/.conda/envs/liana_scanpy/lib/python3.10/site-packages/liana/method/_pipe_utils/_pre.py:149: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
